In [11]:
from openai import OpenAI

URL = "https://hf7yvopqiuwbdm-8000.proxy.runpod.net"
client = OpenAI(
    base_url=f"{URL}/v1",
    api_key="sk-hf7yvopqiuwbdm",
)
MODEL_NAME = "google/gemma-4-31B-it"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        # {"role": "assistant", "content": "답변의 형식, 생각의 흐름"},
        {
            "role": "user",
            "content": "Think harder. More deeply focus on your mind. PDF Parser 를 Python 으로 만들어줘. 실행 가능해야해. 뒤에 설명하지말고 코드만 작성할 것.",
        },
    ],
    temperature=0,
    # max_tokens=9000,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "skip_special_tokens": False,
    },
)

print("=== Reasoning Content ===")
print(response.choices[0].message.reasoning)

print("\n=== Final Answer Content ===")
print(response.choices[0].message.content)


=== Reasoning Content ===
*   Goal: Create a PDF Parser in Python.
    *   Requirement: Must be executable.
    *   Constraint: No explanations, only code.
    *   Tone: "Think harder. More deeply focus on your mind." (Implies a desire for high quality/robustness).

    *   `PyPDF2`: Common, but sometimes struggles with complex layouts.
    *   `pdfminer.six`: Very powerful for text extraction and layout analysis.
    *   `PyMuPDF` (fitz): Extremely fast, high quality, handles images and text well.
    *   `pdfplumber`: Built on `pdfminer.six`, great for tables and structured text.

    *Decision:* `PyMuPDF` (fitz) is generally the most robust and performant for a general-purpose parser. I will use it.

    *   Import `fitz`.
    *   Create a class `PDFParser`.
    *   Method to extract text.
    *   Method to extract images.
    *   Method to extract metadata.
    *   Main block to demonstrate functionality.

    *   Handle file not found errors.
    *   Ensure the code is clean and m

In [8]:
from openai import OpenAI

MODEL_NAME = "Qwen/Qwen3.5-9B"

URL = "https://qdlc9sc32po6bg-8000.proxy.runpod.net"
client = OpenAI(
    base_url=f"{URL}/v1",
    api_key="sk-qdlc9sc32po6bg",
)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {
            "role": "user",
            "content": "Explain what a black hole is in simple terms. Respond in Korean.",
        },
    ],
    temperature=0,
    max_tokens=9000,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        # "skip_special_tokens": False,
    },
)

print("=== Reasoning Content ===")
print(response.choices[0].message.reasoning)

print("\n=== Final Answer Content ===")
print(response.choices[0].message.content)


=== Reasoning Content ===
Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Explain what a black hole is.
    *   Constraint 1: Use simple terms (easy to understand).
    *   Constraint 2: Respond in Korean.

2.  **Determine Key Concepts:**
    *   What is it? (A region in space with very strong gravity).
    *   Why is it called "black"? (Nothing, not even light, can escape).
    *   How is it formed? (Usually from a massive star collapsing).
    *   What happens near it? (Things get sucked in, time slows down).
    *   Analogy (optional but helpful for simplicity).

3.  **Drafting the Content (Internal Monologue/Rough Draft in English):**
    *   A black hole is like a vacuum in space.
    *   It has so much mass in a small space that gravity is super strong.
    *   If you get too close, you can't escape.
    *   Light can't escape either, so it looks black.
    *   Stars die and become black holes.
    *   Don't worry, they are far away.

4.  **Translating and Simplify

In v0.19, CUDA graph memory profiling will be enabled by default (VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1), which more accurately accounts for CUDA graph memory during KV cache allocation. 
To try it now, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=1 and increase --gpu-memory-utilization from 0.9500 to 0.9560 to maintain the same effective KV cache size.

H200, VRAM:144GB
>> 1) GPU KV cache size: 908,688 tokens
    전체 사용 가능한 KV Cache 영역의 사이즈를 토큰으로 나타낸 것
>> 2) Maximum concurrency for 122,880 tokens(120K) per request: 28.81x
    최대 120K 사이즈로 요청이 들어왔을 때 동시에 사용 가능한 요청의 수 = 28.81

1. 256K 모델 최대 지원 Context Length 설정은 가능할까요? 
  908,688 > 256,000 따라서 가능하다.

2. 256K 로 늘렸을 때 동시 요청의 개수는? 
  14x 


GLM-5 754B

필요한 모델 로드 용량 = 754GB
서버 1대 = H200 GPU, 144GB * 8 = 1152GB
KV Cache = 1152 - 754 = 398GB

---

서버1대 = H200 GPU, 144GB * 6 = 864GB
필요한 모델 로드 용량 = 754GB

1. Model Weight 를 분산해서 모델을 올릴 수 없을까?
 > 모델을 GPU 대수만큼 나눠야합니다
 > 모델 = Tensor
 > 모델을 쪼갠다 = Tensor-Parallel = TP
 TP = 6 >> 안됩니다. Attention Head = 64 여서 나누어 떨어지지 않음.
 4장 또는 8장 >> 4장은 부족. 8장이 최소.

 **TP = 8 인 설정이 '최소'**

2. GPU 한대당 Model Weight 은 얼마나 올라가있을까요? 
  144GB * 0.95 - 94.25GB = 42.55GB
  
  Q1) - 1번 요청당 사용할 수 있는 KV Cache 사이즈다?
      - 개별, 

3. 8장짜리 서버 1대로 동시요청 몇개 처리할까요? (max-len 최대)
  > 2개의 요청이 '동시에' 들어왔습니다. 어떤 식으로 처리?
   Continuous Batching

4. 모델이 Gemma4-31B 라면, H200-8장에 모델 몇개 올릴 수 있나요?
  > Gemma4 Weight = 58GB
  16개 => 144GB - 58G * 2 = 28G가 남음 = KV Cache 
         > Max Model len 이 짧습니다.
         > 불만 폭주. 
  
  ( TP = 1, DP = 8 ) * 2
  
  12개 => 8대에 모델 12개

    0,1,2,3,4,5,6,7

    0,1,2,3 - 각각 모델을 올릴거구요. >> TP = 1, DP = 4
    4,5,6,7 - 모델을 좀 촘촘하게... >> TP = 1, DP = 8

  8개  => 144GB - 58G * 1 = 86G가 남음 = KV Cache
         > Max Model len 최대 
         > 넉넉하지만, 다 쓰고 있지않다는 불안감...
  
  TP = 1, DP = 8